# 🧱 Lab: Fix the Broken Repo (4 Seeded Bugs)

**Module: Debugging Lab 1 — Read the Failure** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Repair** an unfamiliar broken repository by applying the four-step debugging decision tree to each failure
2. **Diagnose** a mix of loud failures (tracebacks) and silent failures (wrong output, no error) in the same codebase
3. **Maintain** a debug log that records cause and fix for every bug you encounter
4. **Verify** that all fixes are correct by running a verification script that checks every module

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| Debugging Decision Procedure | A repeatable debugging procedure: reproduce the failure, localise it to one pipeline box, shrink the input, then change one thing at a time until the cause is clear. | mini-walk-debug-tree |
| Import & Environment Failures | — | mini-fix-import-failures |
| Silent Failures | An empty filter result or coerced dtype can silently change what your code is operating on, producing confident, wrong output without any exception or traceback. | mini-fix-data-failures |
| Applying all three to a multi-file repo | — | This lab |

## Prerequisites

- **mini-walk-debug-tree**: The four-step tree — reproduce, localise, shrink, change one thing
- **mini-fix-import-failures**: Diagnosing `ModuleNotFoundError`, `ImportError`, and name shadowing
- **mini-fix-data-failures**: Recognising silent failures — empty filters, coerced dtypes, wrong merge keys


## Scenario

You've just been handed a small Python repository — `broken_repo/` — that processes sales data. The previous developer left in a hurry. The repo has **four seeded bugs** — some crash loudly, some fail silently.

Your job: walk each failure through the debug tree and fix it, one bug at a time, until the verification script passes all checks.

The repo has three modules:

| File | Purpose |
|------|---------|
| `loader.py` | Loads sales records from a CSV-like source |
| `analysis.py` | Computes metrics (total revenue, top product) |
| `report.py` | Formats and returns the final report string |

And one verification script:

| File | Purpose |
|------|---------|
| `verify.py` | Runs 4 checks and prints PASS/FAIL for each |

## Step 0 — Create the Broken Repo

We'll generate the repo files right here in the notebook. In a real scenario, you'd clone a repo — but the debugging process is identical.

In [1]:
import os

# Create the repo directory
repo_dir = "broken_repo"
os.makedirs(repo_dir, exist_ok=True)

print(f"Created directory: {repo_dir}/")

Created directory: broken_repo/


In [2]:
# --- loader.py ---
# BUG 1 (loud):  Wrong import — uses a function name that doesn't exist
# BUG 2 (silent): Quantities stay as strings from the CSV source

loader_code = '''"""Load sales data from a CSV-like source."""

from collections import OrderableDict   # BUG 1: wrong name


def load_sales():
    """Return a list of sales records (simulating a CSV read)."""
    raw_rows = [
        {"product": "Widget",  "quantity": "15", "unit_price": 9.99},
        {"product": "Gadget",  "quantity": "8",  "unit_price": 24.50},
        {"product": "Widget",  "quantity": "22", "unit_price": 9.99},
        {"product": "Gizmo",   "quantity": "5",  "unit_price": 49.99},
        {"product": "Gadget",  "quantity": "12", "unit_price": 24.50},
    ]
    # BUG 2: quantity is left as a string — no int() conversion
    return raw_rows
'''

with open(os.path.join(repo_dir, "loader.py"), "w", encoding="utf-8") as f:
    f.write(loader_code)

print("Wrote loader.py")

Wrote loader.py


In [3]:
# --- analysis.py ---
# BUG 3 (silent): top_product picks the LOWEST revenue instead of the highest

analysis_code = '''"""Compute sales metrics."""


def total_revenue(sales):
    """Return the total revenue across all sales."""
    return sum(row["quantity"] * row["unit_price"] for row in sales)


def top_product(sales):
    """Return the product name with the highest total revenue."""
    revenue_by_product = {}
    for row in sales:
        name = row["product"]
        revenue_by_product[name] = revenue_by_product.get(name, 0) + (
            row["quantity"] * row["unit_price"]
        )
    # BUG 3: min instead of max
    return min(revenue_by_product, key=revenue_by_product.get)
'''

with open(os.path.join(repo_dir, "analysis.py"), "w", encoding="utf-8") as f:
    f.write(analysis_code)

print("Wrote analysis.py")

Wrote analysis.py


In [4]:
# --- report.py ---
# BUG 4 (loud): f-string references a wrong variable name

report_code = '''"""Format the final sales report."""


def build_report(total, top):
    """Return a formatted report string."""
    lines = [
        "=== Sales Report ===",
        f"Total Revenue: ${total:,.2f}",
        f"Top Product:   {top_product_name}",   # BUG 4: should be 'top'
        "===================",
    ]
    return "\\n".join(lines)
'''

with open(os.path.join(repo_dir, "report.py"), "w", encoding="utf-8") as f:
    f.write(report_code)

print("Wrote report.py")

Wrote report.py


In [5]:
# --- verify.py ---
# This is the test harness. It runs 4 checks.

verify_code = '''"""Verification script — run this to check all fixes."""
import sys

results = []


def check(name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append((name, passed))
    msg = f"  [{status}] {name}"
    if detail and not passed:
        msg += f"  —  {detail}"
    print(msg)


print("Running verification...\\n")

# Check 1: loader imports without error
try:
    from loader import load_sales
    check("loader imports", True)
except Exception as e:
    check("loader imports", False, str(e))
    # Can\'t continue if loader fails
    print("\\n❌ Cannot continue — fix loader.py first.")
    sys.exit(1)

sales = load_sales()

# Check 2: quantities are integers (not strings)
qty_types_ok = all(isinstance(row["quantity"], int) for row in sales)
check("quantities are int", qty_types_ok,
      f"got {type(sales[0][\'quantity\']).__name__}")

# Check 3: total_revenue and top_product return correct values
if qty_types_ok:
    from analysis import total_revenue, top_product
    rev = total_revenue(sales)
    top = top_product(sales)
    expected_rev = 15*9.99 + 8*24.50 + 22*9.99 + 5*49.99 + 12*24.50
    check("total revenue correct", abs(rev - expected_rev) < 0.01,
          f"expected {expected_rev:.2f}, got {rev:.2f}")
    # Widget revenue = (15+22)*9.99 = 369.63
    # Gadget revenue = (8+12)*24.50 = 490.00
    # Gizmo  revenue = 5*49.99      = 249.95
    # Top should be Gadget
    check("top product correct", top == "Gadget",
          f"expected Gadget, got {top}")
else:
    check("total revenue correct", False, "skipped — fix quantities first")
    check("top product correct", False, "skipped — fix quantities first")

# Check 4: report builds without error and contains key info
if qty_types_ok:
    try:
        from report import build_report
        report_text = build_report(rev, top)
        has_total = f"${expected_rev:,.2f}" in report_text
        has_top = "Gadget" in report_text
        check("report builds correctly", has_total and has_top,
              f"missing expected content in report")
    except Exception as e:
        check("report builds correctly", False, str(e))
else:
    check("report builds correctly", False, "skipped — fix quantities first")

# Summary
passed = sum(1 for _, p in results if p)
total = len(results)
print(f"\\n{'='*30}")
print(f"  {passed}/{total} checks passed")
if passed == total:
    print("  ✅ All green — repo is fixed!")
else:
    print("  ❌ Keep going — some checks still fail.")
'''

with open(os.path.join(repo_dir, "verify.py"), "w", encoding="utf-8") as f:
    f.write(verify_code)

print("Wrote verify.py")
print("\nRepo structure:")
for fname in sorted(os.listdir(repo_dir)):
    if fname.endswith(".py"):
        print(f"  {repo_dir}/{fname}")

Wrote verify.py

Repo structure:
  broken_repo/analysis.py
  broken_repo/loader.py
  broken_repo/report.py
  broken_repo/verify.py


## Step 1 — Run the Verification Script (Baseline)

Before fixing anything, let's run the verification script to see how many checks fail. This is our **baseline**.

In [6]:
import subprocess
import sys

# Child processes inherit the kernel's encoding. On Windows Jupyter that is often
# cp1252, which cannot print ✅ / ❌. Force UTF-8 for every repo command.
_SUBPROCESS_ENV = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1"}


def run_py(*args):
    """Run a Python command inside the broken repo (UTF-8 safe on Windows)."""
    return subprocess.run(
        [sys.executable, *args],
        cwd=repo_dir,
        capture_output=True,
        text=True,
        encoding="utf-8",
        env=_SUBPROCESS_ENV,
    )


def run_verify():
    """Run the verification script and print its output."""
    result = run_py("verify.py")
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    return result.returncode

run_verify()

Running verification...

  [FAIL] loader imports  —  cannot import name 'OrderableDict' from 'collections' (C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\collections\__init__.py)

❌ Cannot continue — fix loader.py first.



1

The verification script can't even get past the first check — `loader.py` fails to import. Good. We'll fix bugs in dependency order: loader first, then analysis, then report.

## Setup — Debug Log

As in the previous mini-labs, we'll keep a debug log.

In [7]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

debug_log = []

---

## Bug 1 — `loader.py` Won't Import (Loud Failure)

### REPRODUCE

We already saw it fail. Let's reproduce it directly and read the error.

In [8]:
# Reproduce: try to import loader.py
result = run_py("-c", "from loader import load_sales")
print(result.stderr)

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "c:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\lab-repair-broken-repo\broken_repo\loader.py", line 3, in <module>
    from collections import OrderableDict   # BUG 1: wrong name
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ImportError: cannot import name 'OrderableDict' from 'collections' (C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\collections\__init__.py). Did you mean: 'OrderedDict'?



### LOCALISE

The traceback says: `ImportError: cannot import name 'OrderableDict' from 'collections'`. The failing line is the `from collections import OrderableDict` at the top of `loader.py`.

### SHRINK

Already minimal — one import line.

### CHANGE ONE THING

Let's check what `collections` actually provides, then fix the import name. Recall from **mini-fix-import-failures**: use `dir(module)` to find the correct name.

In [9]:
import collections

# What names are actually available?
available = [n for n in dir(collections) if "dict" in n.lower() or "Dict" in n]
print("Dict-related names in collections:")
for name in available:
    print(f"  {name}")

Dict-related names in collections:
  OrderedDict
  UserDict
  _OrderedDictItemsView
  _OrderedDictKeysView
  _OrderedDictValuesView
  defaultdict


The correct name is `OrderedDict`, not `OrderableDict`. Let's fix the file.

Looking at the code more carefully, `OrderedDict` isn't even used in `load_sales()` — the import is dead code. The safest **one change** is to fix the import name so the file loads. (We could also remove the unused import, but we're following the rule: change one thing at a time.)

In [10]:
# Read the current file
loader_path = os.path.join(repo_dir, "loader.py")
with open(loader_path, "r", encoding="utf-8") as f:
    content = f.read()

# The ONE change: fix the import name
content = content.replace("OrderableDict", "OrderedDict")

with open(loader_path, "w", encoding="utf-8") as f:
    f.write(content)

print("Fixed: OrderableDict → OrderedDict")

Fixed: OrderableDict → OrderedDict


In [11]:
# Verify the fix: can we import now?
result = run_py("-c", "from loader import load_sales; print('✅ loader imports OK')")
print(result.stdout.strip())

debug_log.append({
    "bug": "Bug 1",
    "error": "ImportError: cannot import name 'OrderableDict' from 'collections'",
    "cause": "Wrong name — 'OrderableDict' doesn't exist; correct name is 'OrderedDict'",
    "fix": "Changed import to 'OrderedDict'",
})

✅ loader imports OK

In [12]:
# Run verification again to check progress
run_verify()

Running verification...

  [PASS] loader imports
  [FAIL] quantities are int  —  got str
  [FAIL] total revenue correct  —  skipped — fix quantities first
  [FAIL] top product correct  —  skipped — fix quantities first
  [FAIL] report builds correctly  —  skipped — fix quantities first

  1/5 checks passed
  ❌ Keep going — some checks still fail.



0

Check 1 passes now, but Check 2 fails — quantities are strings. That's Bug 2.

---

## Bug 2 — Quantities Are Strings (Silent Failure)

This is a **silent failure** — recall from **mini-fix-data-failures** that CSV data arrives as strings. There's no traceback; the verification script explicitly checks the types.

### REPRODUCE

In [13]:
# Reproduce: load the data and check types
result = run_py(
    "-c",
    "from loader import load_sales; sales = load_sales(); "
    "print(type(sales[0]['quantity']).__name__, repr(sales[0]['quantity']))",
)
print(f"Type and value of first quantity: {result.stdout.strip()}")
print("\n✅ Reproduced! Quantity is a string, not an int.")

Type and value of first quantity: str '15'



✅ Reproduced! Quantity is a string, not an int.


### LOCALISE

The data is defined inside `load_sales()` in `loader.py`. The `quantity` values are literal strings like `"15"`. There is no conversion step.

### SHRINK

Minimal case: `{"quantity": "15"}` — one record, one field.

### CHANGE ONE THING

Add an `int()` conversion for `quantity` before returning the data.

In [14]:
# Read the current file
with open(loader_path, "r", encoding="utf-8") as f:
    content = f.read()

# The ONE change: add int() conversion before the return
old_return = "    return raw_rows"
new_return = """    for row in raw_rows:
        row["quantity"] = int(row["quantity"])
    return raw_rows"""

content = content.replace(old_return, new_return)

with open(loader_path, "w", encoding="utf-8") as f:
    f.write(content)

print("Fixed: added int() conversion for quantity")

Fixed: added int() conversion for quantity


In [15]:
# Verify: are quantities integers now?
result = run_py(
    "-c",
    "from loader import load_sales; sales = load_sales(); "
    "print(type(sales[0]['quantity']).__name__, sales[0]['quantity'])",
)
print(f"Type and value: {result.stdout.strip()}")

debug_log.append({
    "bug": "Bug 2",
    "error": "Silent: quantities are strings — arithmetic would be wrong or crash",
    "cause": "CSV-like source stores everything as strings; no int() conversion",
    "fix": "Added int(row['quantity']) loop before returning data",
})

Type and value: int 15


In [16]:
# Run verification again
run_verify()

Running verification...

  [PASS] loader imports
  [PASS] quantities are int
  [PASS] total revenue correct
  [FAIL] top product correct  —  expected Gadget, got Gizmo
  [FAIL] report builds correctly  —  name 'top_product_name' is not defined

  3/5 checks passed
  ❌ Keep going — some checks still fail.



0

Checks 1–3 should pass now (loader imports, quantities are int, total revenue is correct). But check 4 (`top product correct`) fails — it says we got the **wrong** product. That's Bug 3 — a silent failure.

---

## Bug 3 — Wrong Top Product (Silent Failure)

The verification script expected `Gadget` as the top product, but got something else. The code ran without error. This is the kind of silent bug from **mini-fix-data-failures** where the output is **confident and wrong**.

### REPRODUCE

In [17]:
# Reproduce: what does top_product return?
result = run_py(
    "-c",
    "from loader import load_sales; from analysis import top_product; "
    "sales = load_sales(); print('Top product:', top_product(sales))",
)
print(result.stdout.strip())
print("\n✅ Reproduced! Wrong product returned — no error, just wrong answer.")

Top product: Gizmo



✅ Reproduced! Wrong product returned — no error, just wrong answer.


### LOCALISE

Let's look at the `top_product` function in `analysis.py`.

In [18]:
analysis_path = os.path.join(repo_dir, "analysis.py")
with open(analysis_path, "r", encoding="utf-8") as f:
    print(f.read())

"""Compute sales metrics."""


def total_revenue(sales):
    """Return the total revenue across all sales."""
    return sum(row["quantity"] * row["unit_price"] for row in sales)


def top_product(sales):
    """Return the product name with the highest total revenue."""
    revenue_by_product = {}
    for row in sales:
        name = row["product"]
        revenue_by_product[name] = revenue_by_product.get(name, 0) + (
            row["quantity"] * row["unit_price"]
        )
    # BUG 3: min instead of max
    return min(revenue_by_product, key=revenue_by_product.get)



The last line of `top_product` uses `min(...)` instead of `max(...)`. It returns the **lowest** revenue product instead of the highest.

### SHRINK

Minimal case: two products, different revenues — `min` picks the wrong one.

### CHANGE ONE THING

Replace `min` with `max`.

In [19]:
# Read and fix
with open(analysis_path, "r", encoding="utf-8") as f:
    content = f.read()

# The ONE change: min → max
content = content.replace(
    "return min(revenue_by_product, key=revenue_by_product.get)",
    "return max(revenue_by_product, key=revenue_by_product.get)",
)

with open(analysis_path, "w", encoding="utf-8") as f:
    f.write(content)

print("Fixed: min → max in top_product()")

Fixed: min → max in top_product()


In [20]:
# Verify: does top_product return Gadget now?
result = run_py(
    "-c",
    "from loader import load_sales; from analysis import top_product; "
    "sales = load_sales(); print('Top product:', top_product(sales))",
)
print(result.stdout.strip())

debug_log.append({
    "bug": "Bug 3",
    "error": "Silent: top_product returns lowest-revenue product instead of highest",
    "cause": "Used min() instead of max() to select the top product",
    "fix": "Changed min() to max() in the return statement",
})

Top product: Gadget


In [21]:
# Run verification again
run_verify()

Running verification...

  [PASS] loader imports
  [PASS] quantities are int
  [PASS] total revenue correct
  [PASS] top product correct
  [FAIL] report builds correctly  —  name 'top_product_name' is not defined

  4/5 checks passed
  ❌ Keep going — some checks still fail.



0

Checks 1–4 should pass. But the last check — `report builds correctly` — fails with a `NameError`. That's Bug 4.

---

## Bug 4 — Report Crashes with `NameError` (Loud Failure)

### REPRODUCE

In [22]:
# Reproduce: try to build a report
result = run_py("-c", "from report import build_report; print(build_report(1000.0, 'Gadget'))")
if result.returncode != 0:
    print("✅ Reproduced!")
    # Show just the last two lines of the traceback
    lines = result.stderr.strip().split("\n")
    for line in lines[-3:]:
        print(f"  {line}")

✅ Reproduced!
      f"Top Product:   {top_product_name}",   # BUG 4: should be 'top'
                        ^^^^^^^^^^^^^^^^
  NameError: name 'top_product_name' is not defined


### LOCALISE

The error is `NameError: name 'top_product_name' is not defined`. Let's look at `report.py`.

In [23]:
report_path = os.path.join(repo_dir, "report.py")
with open(report_path, "r", encoding="utf-8") as f:
    print(f.read())

"""Format the final sales report."""


def build_report(total, top):
    """Return a formatted report string."""
    lines = [
        "=== Sales Report ===",
        f"Total Revenue: ${total:,.2f}",
        f"Top Product:   {top_product_name}",   # BUG 4: should be 'top'
        "===================",
    ]
    return "\n".join(lines)



The f-string uses `top_product_name` but the parameter is called `top`. The developer probably renamed the parameter and forgot to update the f-string.

### SHRINK

Already minimal: `build_report(1000.0, 'Gadget')` is enough to trigger it.

### CHANGE ONE THING

Replace `top_product_name` with `top` in the f-string.

In [24]:
# Read and fix
with open(report_path, "r", encoding="utf-8") as f:
    content = f.read()

# The ONE change: fix the variable reference
content = content.replace("top_product_name", "top")

with open(report_path, "w", encoding="utf-8") as f:
    f.write(content)

print("Fixed: top_product_name → top")

Fixed: top_product_name → top


In [25]:
# Verify: does the report build now?
result = run_py("-c", "from report import build_report; print(build_report(1000.0, 'Gadget'))")
print(result.stdout.strip())

debug_log.append({
    "bug": "Bug 4",
    "error": "NameError: name 'top_product_name' is not defined",
    "cause": "f-string references 'top_product_name' but the parameter is named 'top'",
    "fix": "Changed 'top_product_name' to 'top' in the f-string",
})

=== Sales Report ===
Total Revenue: $1,000.00
Top Product:   Gadget

---

## Final Verification — All Green?

Let's run the verification script one final time. All 5 checks should pass.

In [26]:
return_code = run_verify()

Running verification...

  [PASS] loader imports
  [PASS] quantities are int
  [PASS] total revenue correct
  [PASS] top product correct
  [PASS] report builds correctly

  5/5 checks passed
  ✅ All green — repo is fixed!



---

## Review the Debug Log

Here's every bug we fixed, recorded in order.

In [27]:
rows = "| # | Bug | Error | Cause | Fix |\n|---|-----|-------|-------|-----|\n"
for i, entry in enumerate(debug_log, 1):
    rows += f"| {i} | {entry['bug']} | {entry['error'][:55]} | {entry['cause']} | {entry['fix']} |\n"

md(rows)

| # | Bug | Error | Cause | Fix |
|---|-----|-------|-------|-----|
| 1 | Bug 1 | ImportError: cannot import name 'OrderableDict' from 'c | Wrong name — 'OrderableDict' doesn't exist; correct name is 'OrderedDict' | Changed import to 'OrderedDict' |
| 2 | Bug 2 | Silent: quantities are strings — arithmetic would be wr | CSV-like source stores everything as strings; no int() conversion | Added int(row['quantity']) loop before returning data |
| 3 | Bug 3 | Silent: top_product returns lowest-revenue product inst | Used min() instead of max() to select the top product | Changed min() to max() in the return statement |
| 4 | Bug 4 | NameError: name 'top_product_name' is not defined | f-string references 'top_product_name' but the parameter is named 'top' | Changed 'top_product_name' to 'top' in the f-string |


## Cleanup

In [28]:
import shutil

shutil.rmtree(repo_dir, ignore_errors=True)
print(f"Cleaned up {repo_dir}/")

Cleaned up broken_repo/


## 🎯 Summary

### What You Built

You repaired an unfamiliar broken repository with four seeded bugs — two loud crashes and two silent failures — by walking each bug through the four-step debugging decision tree until the verification script passed all checks.

### Key Takeaways

1. **Start with the verification script** — running the test suite first tells you how many bugs exist and which to fix in what order (dependency order)
2. **Loud failures first** — import errors and `NameError`s block everything downstream; fix them before hunting silent bugs
3. **Silent failures need a known-good answer** — for Bug 2 (string quantities) and Bug 3 (min vs max), the only way to "reproduce" was to compare actual output against expected values
4. **One change at a time, re-verify** — after each fix we re-ran the verification script; this confirmed the fix worked and revealed the next bug without confusion
5. **The debug log is your audit trail** — recording error → cause → fix for each bug makes the repair process reviewable and the knowledge reusable